Parses Bronze into typed staging tables (Day 6).
Reads:  jobmarket.bronze.job_postings_api, jobmarket.bronze.job_postings_kaggle
Writes: jobmarket.silver.stg_postings_api, jobmarket.silver.stg_postings_kaggle
Staging = parsed + typed, NOT yet normalized/deduped (Days 7-10).

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, DoubleType, ArrayType
)

# Schema = a tree of StructField(name, type, nullable)
# We declare ONLY the fields we need — from_json ignores the rest.
adzuna_schema = StructType([
    StructField("id",                  StringType(), True),
    StructField("title",               StringType(), True),
    StructField("description",         StringType(), True),
    StructField("created",             StringType(), True),   # ISO timestamp, cast later
    StructField("salary_min",          DoubleType(), True),
    StructField("salary_max",          DoubleType(), True),
    StructField("salary_is_predicted", StringType(), True),   # the '1'/'0' STRING quirk!
    # nested objects = StructType inside StructType
    StructField("company", StructType([
        StructField("display_name", StringType(), True)
    ]), True),
    StructField("location", StructType([
        StructField("display_name", StringType(), True),
        StructField("area", ArrayType(StringType()), True)    # variable-length array
    ]), True),
])

In [0]:
bronze_api = spark.table("jobmarket.bronze.job_postings_api")

parsed = bronze_api.withColumn(
    "j", F.from_json("raw_payload", adzuna_schema)   # text -> structured column
)

stg_api = parsed.select(
    F.col("j.id").alias("source_posting_id"),
    F.col("j.title").alias("title_raw"),
    F.col("j.company.display_name").alias("company"),      # dot path into nested struct

  # area array: variable length 1-4 elements
    # ['US'] | ['US','Colorado','Denver'] | ['US','Illinois','Cook County','Chicago']
    # F.get() returns NULL for out-of-bounds instead of erroring (ANSI-safe)
    F.get("j.location.area", 1).alias("state"),

    # last element = city; size-1 computes the last index per row
    F.when(F.size("j.location.area") >= 3,
           F.get("j.location.area", F.size("j.location.area") - 1)
    ).alias("city"),

    F.col("j.salary_min").alias("salary_min"),
    F.col("j.salary_max").alias("salary_max"),

    # the defused trap: string comparison, NOT boolean cast
    (F.col("j.salary_is_predicted") == "1").alias("salary_is_estimated"),

    F.col("j.description").alias("description"),

    # ISO string -> timestamp -> date
    F.to_date(F.to_timestamp("j.created")).alias("posted_date"),

    # crude v1 remote detection: word appears in title or description
    (F.lower("j.title").contains("remote") |
     F.lower("j.description").contains("remote")).alias("is_remote"),

    F.col("source"),
    F.col("ingest_ts"),
)

display(stg_api.limit(10))

In [0]:
(stg_api.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("jobmarket.silver.stg_postings_api"))

In [0]:
total = stg_api.count()

checks = stg_api.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c + "_nulls")
    for c in ["source_posting_id", "title_raw", "company", "city",
              "state", "posted_date"]
])
print("Total rows:", total)
checks.display()

In [0]:
parsed.select(F.size("j.location.area").alias("n")) \
      .groupBy("n").count().orderBy("n").display()

In [0]:
spark.table("jobmarket.bronze.job_postings_kaggle").printSchema()

In [0]:
bronze_kaggle = spark.table("jobmarket.bronze.job_postings_kaggle")

stg_kaggle = bronze_kaggle.select(
    F.col("job_id").alias("source_posting_id"),
    F.col("title").alias("title_raw"),
    F.col("company_name").alias("company"),

    # defensive split from the start — no comma = null state, no error
    F.trim(F.get(F.split("location", ","), 0)).alias("city"),
    F.trim(F.get(F.split("location", ","), 1)).alias("state"),

    # salaries: cast to double but keep _raw naming — Day 8 annualizes
    F.col("min_salary").cast("double").alias("salary_min_raw"),
    F.col("max_salary").cast("double").alias("salary_max_raw"),
    F.col("pay_period"),

    F.col("description"),

    # listed_time is a STRING of epoch milliseconds -> cast, divide, convert
    F.to_date(F.from_unixtime(F.col("listed_time").cast("double") / 1000))
        .alias("posted_date"),

    # bonus columns the dataset gives us — cheaper than deriving ourselves:
    F.coalesce(F.col("remote_allowed") == "1.0", F.lit(False)).alias("is_remote"),         # verify 0/1 in display!
    F.col("formatted_experience_level").alias("experience_level"), # Day 7 seniority input

    F.col("source"),
    F.col("ingest_ts"),
)

display(stg_kaggle.limit(10))

In [0]:
spark.table("jobmarket.bronze.job_postings_kaggle") \
     .groupBy("remote_allowed").count().display()

In [0]:
(stg_kaggle.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("jobmarket.silver.stg_postings_kaggle"))

In [0]:
stg_kaggle.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c + "_nulls")
    for c in ["source_posting_id", "title_raw", "company", "posted_date", "description"]
]).display()